# 13 — Prepare all-class canonical RGB + MediaPipe data

Notebook này tạo một data contract mới cho **toàn bộ từ canonical đủ điều kiện** của bộ Kaggle VSL: mỗi lớp phải có ít nhất 2 mẫu official-train và ít nhất 1 mẫu official-test. Validation được tách xác định từ official-train; release không có signer ID nên **không tuyên bố signer-disjoint**.

Pose dùng lại archive MediaPipe `[T,76,3]` của notebook 11. RGB chỉ lấy `processed/processed/frame_splited`, không tải 72 GB video raw và không lấy `processed_augmented`. Frozen VideoMAE V2 được chạy đúng một lần, lưu temporal tokens `[8,768]` dạng float16 theo shard có resume trên Drive.


In [ ]:
#@title Configuration
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
KAGGLE_DATASET = 'nguyenanfms/vsl-vietnamese-sign-language-v2'  #@param {type:'string'}
KAGGLE_VERSION = 0  #@param {type:'integer'}
MIN_OFFICIAL_TRAIN_SAMPLES = 2  #@param {type:'integer'}
VALIDATION_FRACTION = 0.20  #@param {type:'number'}
SEED = 42  #@param {type:'integer'}
RGB_MODEL_ID = 'OpenGVLab/VideoMAEv2-Base'  #@param {type:'string'}
RGB_MODEL_REVISION = '0e826d7e85e39f9d951e331cd91c5c2d8142d385'  #@param {type:'string'}
RGB_BATCH_SIZE = 2  #@param {type:'integer'}
RGB_SHARD_SIZE = 256  #@param {type:'integer'}
DOWNLOAD_WORKERS = 2  #@param {type:'integer'}
SAVE_RGB_ARCHIVE_TO_DRIVE = True  #@param {type:'boolean'}


In [ ]:
#@title Mount Drive and define artifacts
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results')
POSE_SOURCE_ROOT = DRIVE_ROOT / 'vsl_kaggle_mediapipe/source'
RESULTS_ROOT = DRIVE_ROOT / 'vsl_kaggle_rgb_pose'
VERSION_TAG = 'latest' if KAGGLE_VERSION == 0 else f'v{KAGGLE_VERSION}'
fraction_tag = f'{VALIDATION_FRACTION:.4f}'.rstrip('0').rstrip('.').replace('.', 'p')
DATA_CONTRACT_NAME = f'all_eligible_{VERSION_TAG}_min{MIN_OFFICIAL_TRAIN_SAMPLES}_val{fraction_tag}_seed{SEED}'
SUBSET_ROOT = RESULTS_ROOT / 'subsets' / DATA_CONTRACT_NAME
SOURCE_ROOT = RESULTS_ROOT / 'source'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
BUILD_REPORT = SUBSET_ROOT / 'manifest_report.json'
POSE_PACK = SUBSET_ROOT / 'pose/mediapipe76_front.npz'
POSE_PACK_REPORT = SUBSET_ROOT / 'pose/mediapipe76_front.report.json'
RGB_FETCH_REPORT = SUBSET_ROOT / 'rgb/fetch_report.json'
RGB_MANIFEST = SUBSET_ROOT / 'rgb/manifest_rgb.csv'
RGB_FEATURES = SUBSET_ROOT / 'rgb/videomaev2_base_tokens_fp16.npz'
RGB_FEATURE_REPORT = SUBSET_ROOT / 'rgb/videomaev2_base_tokens_fp16.report.json'
RGB_SHARDS = SUBSET_ROOT / 'rgb/videomaev2_base_shards'
KEYPOINT_ARCHIVE = POSE_SOURCE_ROOT / f'keypoints_splited_all_canonical_{VERSION_TAG}.tar'
KEYPOINT_ARCHIVE_REPORT = POSE_SOURCE_ROOT / f'keypoints_splited_all_canonical_{VERSION_TAG}.json'
LOCAL_REPO = Path('/content/silent-signal')
LOCAL_KEYPOINT_ROOT = Path('/content/kaggle-vsl-keypoints-all-canonical')
LOCAL_RGB_ROOT = Path('/content/kaggle-vsl-rgb-all-eligible')
for directory in (SUBSET_ROOT, SOURCE_ROOT, POSE_PACK.parent, RGB_FEATURES.parent):
    directory.mkdir(parents=True, exist_ok=True)
print('Drive subset root:', SUBSET_ROOT)


In [ ]:
#@title Checkout the pinned branch and install runtime
import subprocess, sys
if not (LOCAL_REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
                    'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF,
                    f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{LOCAL_REPO}[training]',
                'opencv-python-headless', 'transformers==4.57.6', 'timm>=1.0,<2', 'easydict'], check=True)
commit = subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', commit)


In [ ]:
#@title Restore and validate all canonical MediaPipe keypoints from Drive
import json, shutil, tarfile
if not KEYPOINT_ARCHIVE.is_file() or not KEYPOINT_ARCHIVE_REPORT.is_file():
    raise FileNotFoundError('Run notebook 11 once to create the canonical keypoint tar + report.')
keypoint_marker = json.loads(KEYPOINT_ARCHIVE_REPORT.read_text(encoding='utf-8'))
if keypoint_marker.get('complete') is not True:
    raise RuntimeError(f'Notebook 11 archive is not marked complete: {keypoint_marker}')
expected_dataset_handle = KAGGLE_DATASET if KAGGLE_VERSION == 0 else f'{KAGGLE_DATASET}/versions/{KAGGLE_VERSION}'
if keypoint_marker.get('dataset_handle') != expected_dataset_handle:
    raise RuntimeError('Notebook 11 keypoints were created from a different Kaggle dataset/version.')
expected_npy = int(keypoint_marker['source_files'])
expected_bytes = int(keypoint_marker['source_bytes'])
def keypoint_tree_identity(root):
    files = tuple(root.rglob('*.npy')) if root.is_dir() else ()
    return len(files), sum(path.stat().st_size for path in files)
if keypoint_tree_identity(LOCAL_KEYPOINT_ROOT) != (expected_npy, expected_bytes):
    staging = LOCAL_KEYPOINT_ROOT.with_name(LOCAL_KEYPOINT_ROOT.name + '.partial')
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    print('Restoring verified keypoint archive:', KEYPOINT_ARCHIVE)
    with tarfile.open(KEYPOINT_ARCHIVE, 'r') as archive:
        archive.extractall(staging, filter='data')
    actual = keypoint_tree_identity(staging)
    if actual != (expected_npy, expected_bytes):
        raise RuntimeError(f'Incomplete keypoint restore: expected {(expected_npy, expected_bytes)}, found {actual}')
    shutil.rmtree(LOCAL_KEYPOINT_ROOT, ignore_errors=True)
    staging.replace(LOCAL_KEYPOINT_ROOT)
print(f'Canonical keypoints verified: {expected_npy:,} files, {expected_bytes / 1024**3:.2f} GiB')


In [ ]:
#@title Build the all-eligible manifest and pack matching pose arrays
import json
DATASET_HANDLE = KAGGLE_DATASET if KAGGLE_VERSION == 0 else f'{KAGGLE_DATASET}/versions/{KAGGLE_VERSION}'
build = [sys.executable, '-m', 'silent_signal.cli.prepare_kaggle_vsl_mediapipe', 'build',
         '--keypoint-root', str(LOCAL_KEYPOINT_ROOT), '--manifest', str(MANIFEST),
         '--labels', str(LABELS), '--selection', str(SELECTION), '--report', str(BUILD_REPORT),
         '--classes', '0', '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
         '--validation-fraction', str(VALIDATION_FRACTION), '--seed', str(SEED),
         '--dataset-handle', DATASET_HANDLE]
print('+', ' '.join(build), flush=True)
subprocess.run(build, check=True)
pack = [sys.executable, '-m', 'silent_signal.cli.prepare_kaggle_vsl_mediapipe', 'pack',
        '--keypoint-root', str(LOCAL_KEYPOINT_ROOT), '--manifest', str(MANIFEST),
        '--output', str(POSE_PACK), '--report', str(POSE_PACK_REPORT),
        '--dataset-handle', DATASET_HANDLE, '--progress-every', '250']
print('+', ' '.join(pack), flush=True)
subprocess.run(pack, check=True)
manifest_report = json.loads(BUILD_REPORT.read_text(encoding='utf-8'))
print('Actual eligible classes:', manifest_report['classes'])
print('Split counts:', manifest_report['splits'])
print('Warning:', manifest_report['protocol_warning'])


In [ ]:
#@title Reuse a verified RGB cache, otherwise restore/resume canonical MP4s
import os, shutil
from silent_signal.cli.extract_videomae_features import completed_output_is_current, extraction_fingerprint
from silent_signal.data.manifest import read_manifest
from silent_signal.data.rgb_feature_pack import read_rgb_feature_pack
from silent_signal.pose.cache import sha256_file
records = tuple(sorted(read_manifest(MANIFEST), key=lambda row: row.sample_id))
manifest_sha = sha256_file(MANIFEST)
identity = extraction_fingerprint(manifest_sha, model_id=RGB_MODEL_ID, model_revision=RGB_MODEL_REVISION)
expected_ids = tuple(row.sample_id for row in records)
RGB_CACHE_READY = completed_output_is_current(
    RGB_FEATURES, RGB_FEATURE_REPORT, sample_ids=expected_ids, extraction_identity=identity
)
print('Complete RGB feature cache ready:', RGB_CACHE_READY)

RGB_ARCHIVE = SOURCE_ROOT / f'frame_splited_{DATA_CONTRACT_NAME}_{manifest_sha[:12]}.tar'
RGB_ARCHIVE_REPORT = RGB_ARCHIVE.with_suffix('.json')
def rgb_tree_identity(root):
    files = tuple(root.rglob('*.mp4')) if root.is_dir() else ()
    return len(files), sum(path.stat().st_size for path in files)

RGB_SOURCE_READY = False
rgb_source_report = None
if not RGB_CACHE_READY and RGB_ARCHIVE.is_file() and RGB_ARCHIVE_REPORT.is_file():
    marker = json.loads(RGB_ARCHIVE_REPORT.read_text(encoding='utf-8'))
    marker_ok = (marker.get('complete') is True and marker.get('manifest_sha256') == manifest_sha
                 and int(marker.get('files', -1)) == len(records))
    expected_tree = (int(marker.get('files', -1)), int(marker.get('selected_bytes', -1)))
    if marker_ok and rgb_tree_identity(LOCAL_RGB_ROOT) != expected_tree:
        staging = LOCAL_RGB_ROOT.with_name(LOCAL_RGB_ROOT.name + '.partial')
        shutil.rmtree(staging, ignore_errors=True); staging.mkdir(parents=True)
        print('Restoring verified RGB archive:', RGB_ARCHIVE)
        with tarfile.open(RGB_ARCHIVE, 'r') as archive:
            archive.extractall(staging, filter='data')
        if rgb_tree_identity(staging) != expected_tree:
            raise RuntimeError('RGB archive restore is incomplete; delete its tar + json marker and rerun.')
        shutil.rmtree(LOCAL_RGB_ROOT, ignore_errors=True); staging.replace(LOCAL_RGB_ROOT)
    RGB_SOURCE_READY = marker_ok and rgb_tree_identity(LOCAL_RGB_ROOT) == expected_tree
    rgb_source_report = marker if RGB_SOURCE_READY else None

if not RGB_CACHE_READY and not RGB_SOURCE_READY:
    from google.colab import userdata
    def secret(name):
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''
    username, key, token = secret('KAGGLE_USERNAME'), secret('KAGGLE_KEY'), secret('KAGGLE_API_TOKEN')
    if username and key:
        os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = username, key
        os.environ.pop('KAGGLE_API_TOKEN', None)
    elif token:
        os.environ['KAGGLE_API_TOKEN'] = token
    else:
        raise RuntimeError('Enable KAGGLE_USERNAME + KAGGLE_KEY or KAGGLE_API_TOKEN in Colab Secrets.')
    fetch = [sys.executable, '-m', 'silent_signal.cli.fetch_kaggle_vsl_rgb',
             '--manifest', str(MANIFEST), '--output-root', str(LOCAL_RGB_ROOT),
             '--rgb-manifest', str(RGB_MANIFEST), '--report', str(RGB_FETCH_REPORT),
             '--workers', str(DOWNLOAD_WORKERS), '--dataset', KAGGLE_DATASET,
             '--version', str(KAGGLE_VERSION)]
    print('+', ' '.join(fetch), flush=True)
    subprocess.run(fetch, check=True)
    rgb_source_report = json.loads(RGB_FETCH_REPORT.read_text(encoding='utf-8'))
    expected_tree = (int(rgb_source_report['files']), int(rgb_source_report['selected_bytes']))
    RGB_SOURCE_READY = (rgb_source_report.get('status') == 'complete'
                        and rgb_source_report.get('manifest_sha256') == manifest_sha
                        and expected_tree == (len(records), expected_tree[1])
                        and rgb_tree_identity(LOCAL_RGB_ROOT) == expected_tree)
    if not RGB_SOURCE_READY:
        raise RuntimeError('RGB fetch finished without a complete exact-manifest inventory.')

if not RGB_CACHE_READY:
    if not RGB_FETCH_REPORT.is_file():
        RGB_FETCH_REPORT.write_text(json.dumps(rgb_source_report, ensure_ascii=False, indent=2), encoding='utf-8')
    pose_source_report = json.loads((LOCAL_KEYPOINT_ROOT / '_fetch_report.json').read_text(encoding='utf-8'))
    for key in ('archive_size', 'archive_etag'):
        left, right = pose_source_report.get(key), rgb_source_report.get(key)
        if left is not None and right is not None and left != right:
            raise RuntimeError(f'Pose and RGB came from different Kaggle archives ({key}: {left} != {right}).')

if not RGB_CACHE_READY and SAVE_RGB_ARCHIVE_TO_DRIVE:
    marker = {**rgb_source_report, 'complete': True, 'manifest_sha256': manifest_sha}
    current_marker = json.loads(RGB_ARCHIVE_REPORT.read_text(encoding='utf-8')) if RGB_ARCHIVE_REPORT.is_file() else {}
    archive_ok = (RGB_ARCHIVE.is_file() and current_marker.get('complete') is True
                  and current_marker.get('manifest_sha256') == manifest_sha
                  and int(current_marker.get('files', -1)) == len(records)
                  and int(current_marker.get('archive_bytes', -1)) == RGB_ARCHIVE.stat().st_size)
    if not archive_ok:
        partial = RGB_ARCHIVE.with_suffix('.tar.partial'); partial.unlink(missing_ok=True)
        print('Saving one verified RGB archive to Drive:', RGB_ARCHIVE)
        with tarfile.open(partial, 'w') as archive:
            for path in sorted(LOCAL_RGB_ROOT.rglob('*.mp4')):
                archive.add(path, arcname=path.relative_to(LOCAL_RGB_ROOT), recursive=False)
        partial.replace(RGB_ARCHIVE)
        marker['archive_bytes'] = RGB_ARCHIVE.stat().st_size
        RGB_ARCHIVE_REPORT.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'RGB archive ready: {RGB_ARCHIVE.stat().st_size / 1024**3:.2f} GiB')


In [ ]:
#@title Extract frozen VideoMAE temporal tokens — resumable shards
if RGB_CACHE_READY:
    print('Reusing the verified final RGB feature pack; no MP4/model work is needed.')
else:
    extract = [sys.executable, '-u', '-m', 'silent_signal.cli.extract_videomae_features',
               '--manifest', str(MANIFEST), '--dataset-root', str(LOCAL_RGB_ROOT),
               '--output', str(RGB_FEATURES), '--report', str(RGB_FEATURE_REPORT),
               '--shard-root', str(RGB_SHARDS), '--model-id', RGB_MODEL_ID,
               '--model-revision', RGB_MODEL_REVISION, '--batch-size', str(RGB_BATCH_SIZE),
               '--shard-size', str(RGB_SHARD_SIZE), '--progress-every', '10', '--device', 'cuda']
    print('+', ' '.join(extract), flush=True)
    subprocess.run(extract, check=True)


In [ ]:
#@title Verify exact pose/RGB alignment and print hand-off
from collections import Counter
from silent_signal.data.keypoint_pack import read_packed_keypoints
pose = read_packed_keypoints(POSE_PACK)
rgb = read_rgb_feature_pack(RGB_FEATURES, expected_fingerprint=identity)
expected = tuple(row.sample_id for row in records)
assert tuple(sorted(pose.sample_ids)) == expected
assert rgb.sample_ids == expected
assert rgb.features.shape[1:] == (8, 768)
print('Alignment: PASS')
print('Classes:', len({row.class_index for row in records}))
print('Samples:', len(records), dict(Counter(str(row.split) for row in records)))
print('Pose pack:', POSE_PACK)
print('RGB feature pack:', RGB_FEATURES, rgb.features.shape, rgb.features.dtype)
print('Notebook 14 can now train without MP4 files or the VideoMAE backbone.')


## Notes

- `all_eligible` nghĩa là mọi từ canonical có thể tạo train + validation + official test; số lớp thực tế được in sau bước build, không ép cứng là 472.
- Chạy notebook từ trên xuống dưới. Tên data contract tự chứa ngưỡng mẫu, tỉ lệ validation và seed để không trộn các lần chia dữ liệu.
- `MIN_OFFICIAL_TRAIN_SAMPLES=2` giữ tối đa vốn từ nhưng lớp ít mẫu sẽ có phương sai cao và dễ overfit; có thể tăng ngưỡng để đổi lấy đánh giá ổn định hơn.
- RGB downloader gom các entry gần nhau thành range lớn và có CRC resume, tránh hàng chục nghìn request riêng lẻ.
- Shard RGB nằm trên Drive nên khi Colab ngắt, chạy lại notebook sẽ bỏ qua shard đã hoàn tất.
- VideoMAE V2 Base được pin theo commit và dùng cho nghiên cứu/giáo dục theo giấy phép của model.
- Nếu Drive không đủ chỗ cho tar MP4, tắt `SAVE_RGB_ARCHIVE_TO_DRIVE`; cache token vẫn là artifact bắt buộc cho notebook 14.
